# Export Gold tables to Parquet for Power BI

This notebook reads the current four Gold Delta tables, exports one standalone Parquet file per table, and reads each export back to verify row count, column names and types, and revenue. The files are local reporting copies, not Delta tables. Run after all four Gold notebooks.

Power BI Desktop: Get data > Parquet > choose each file in powerbi/exports. The date and money types should remain typed. Do not add revenue from different marts together, because the same sales are represented at different grains.


In [ ]:
import os  # Set the correct Java installation for WSL Spark.
import shutil  # Find Java.
import tempfile  # Create safe temporary export folders.
from pathlib import Path  # Build portable project paths.
from pyspark.sql import SparkSession, functions as F  # Read and validate tables.
from delta import configure_spark_with_delta_pip  # Load Delta support.
java = shutil.which("java")  # Find Linux Java, not Windows Java.
if java is None:  # Check the runtime.
    raise RuntimeError("Linux Java is unavailable")  # Explain startup failure.
os.environ["JAVA_HOME"] = str(Path(java).resolve().parent.parent)  # Point Spark to Linux Java.
project_root = Path("/mnt/f/Big Data/Retail_Lakehouse_Learning")  # Locate the project in WSL.
if not project_root.is_dir():  # Confirm the mounted project exists.
    raise FileNotFoundError(project_root)  # Stop on a wrong path.
builder = (SparkSession.builder.master("local[2]")  # Limit local resource use.
    .appName("RetailGoldParquetExport")  # Label this export.
    .config("spark.sql.shuffle.partitions", "4")  # Use small local shuffles.
    .config("spark.sql.session.timeZone", "Asia/Karachi")  # Keep dates consistent with Gold.
    .config("spark.ui.enabled", "false")  # Avoid a second local Spark UI.
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")  # Enable Delta reads.
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog"))  # Register Delta.
spark = configure_spark_with_delta_pip(builder).getOrCreate()  # Start Spark once for all exports.
spark.sparkContext.setLogLevel("ERROR")  # Keep notebook output focused on checks.
print("Spark:", spark.version, "Timezone:", spark.conf.get("spark.sql.session.timeZone"))  # Show session settings.


In [ ]:
gold_root = project_root / "lakehouse" / "gold" / "data"  # Find the validated Delta tables.
exports_root = project_root / "powerbi" / "exports"  # Choose local Power BI files.
exports_root.mkdir(parents=True, exist_ok=True)  # Create the export folder if needed.
table_specs = [  # Pair each Gold table with the revenue column used to reconcile it.
    ("daily_store_sales", "revenue"),  # Daily sales uses net order revenue.
    ("product_performance", "line_revenue"),  # Product sales uses allocated line revenue.
    ("customer_sales", "revenue"),  # Customer sales uses net order revenue.
    ("weather_sales_analysis", "revenue"),  # Weather view carries the daily revenue.
]  # Keep all four in the requested export set.
daily_df = spark.read.format("delta").load(str(gold_root / "daily_store_sales"))  # Read the control mart.
expected_revenue = daily_df.agg(F.sum("revenue").alias("total")).first().total  # Set the cross-mart control total.
if expected_revenue is None:  # Require a populated control total.
    raise ValueError("Gold daily revenue is missing")  # Stop an incomplete export.
print("Expected revenue:", expected_revenue)  # Display the control figure.


In [ ]:
results = []  # Collect the verification outcome for each exported table.
for table_name, revenue_column in table_specs:  # Export each Gold mart in order.
    source_path = gold_root / table_name  # Choose the current Delta table.
    source_df = spark.read.format("delta").load(str(source_path))  # Load committed Delta rows.
    source_count = source_df.count()  # Record source row count.
    source_total = source_df.agg(F.sum(revenue_column).alias("total")).first().total  # Record source money.
    if source_count == 0 or source_total != expected_revenue:  # Require populated, reconciled inputs.
        raise ValueError(f"{table_name}: Gold source count or revenue failed validation")  # Stop before replacement.
    final_file = exports_root / f"{table_name}.parquet"  # Give Power BI a stable single-file path.
    with tempfile.TemporaryDirectory(prefix=f".stage_{table_name}_", dir=exports_root) as scratch:  # Stage a fresh copy.
        scratch_path = Path(scratch).resolve()  # Resolve the temporary folder.
        if not scratch_path.is_relative_to(exports_root.resolve()):  # Verify it stays inside exports.
            raise ValueError("Temporary export path escaped the export folder")  # Prevent unsafe file work.
        stage_dir = scratch_path / "data"  # Spark writes a directory of part files.
        source_df.coalesce(1).write.mode("errorifexists").parquet(str(stage_dir))  # Write one compressed Parquet part.
        part_files = list(stage_dir.glob("part-*.parquet"))  # Locate Spark's data file.
        if len(part_files) != 1:  # One source table must produce one standalone export file.
            raise ValueError(f"{table_name}: expected one Parquet part, found {len(part_files)}")  # Stop ambiguous output.
        staged_df = spark.read.parquet(str(part_files[0]))  # Read the staged standalone file.
        staged_count = staged_df.count()  # Count records after serialization.
        staged_total = staged_df.agg(F.sum(revenue_column).alias("total")).first().total  # Recheck money.
        source_types = [(field.name, field.dataType.simpleString()) for field in source_df.schema.fields]  # Record source names and types.
        staged_types = [(field.name, field.dataType.simpleString()) for field in staged_df.schema.fields]  # Record Parquet names and types.
        if staged_count != source_count or staged_total != source_total or staged_types != source_types:  # Verify the copy.
            raise ValueError(f"{table_name}: staged Parquet differs from Gold Delta")  # Keep previous final file.
        os.replace(part_files[0], final_file)  # Publish the verified file at a stable path.
    saved_df = spark.read.parquet(str(final_file))  # Reopen the Power BI-facing file.
    saved_count = saved_df.count()  # Verify the published row count.
    saved_total = saved_df.agg(F.sum(revenue_column).alias("total")).first().total  # Verify published money.
    if saved_count != source_count or saved_total != source_total:  # Check after the move.
        raise ValueError(f"{table_name}: final Parquet differs from Gold Delta")  # Report publication failure.
    results.append((table_name, saved_count, str(final_file), final_file.stat().st_size))  # Save summary data.
    print(f"{table_name}: {saved_count} rows, revenue {saved_total}, {final_file.stat().st_size:,} bytes")  # Show one result.


In [ ]:
for table_name, row_count, file_path, byte_count in results:  # Review all four exported files.
    print(f"{table_name}: {row_count} rows | {byte_count:,} bytes | {file_path}")  # Show Power BI paths.
print("All four Parquet exports match their Gold Delta sources.")  # Confirm the full set passed.
